# 2. Model Training

Train Splink model with 9 comparisons, 4 blocking rules, 5-stage EM training.


In [1]:
import sys
sys.path.insert(0, '/Users/robertlalani/Desktop/entity_resolution_12_18_25/1-30-26')

import pandas as pd
import numpy as np
from splink import Linker, SettingsCreator, block_on, DuckDBAPI
import splink.comparison_library as cl
import splink.comparison_level_library as cll

from config import config
from utils import DatabaseManager, log_step, Timer, save_checkpoint, load_checkpoint, describe_dataframe
from data_prep import (
    load_and_prepare_training_data, create_ground_truth_pairs, create_blocking_keys,
    add_idf_based_features, add_token_set_features
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print("Imports loaded")


Imports loaded


In [2]:
print("TRAINING CONFIGURATION")
print("=" * 50)
print(f"dim_org sample size:    {config.sampling.TRAINING_DIM_ORG_SAMPLE:,}")
print(f"GRID sample size:       {config.sampling.TRAINING_GRID_SAMPLE:,}")
print(f"Prediction threshold:   {config.matching.THRESHOLD_PREDICTION}")
print(f"Model output path:      {config.paths.MODEL_FILE}")

db = DatabaseManager()
print("Database ready")


TRAINING CONFIGURATION
dim_org sample size:    110,000
GRID sample size:       110,000
Prediction threshold:   0.5
Model output path:      /Users/robertlalani/Desktop/entity_resolution_12_18_25/1-30-26/models/model_v1.json
Database ready


In [3]:
# Load training data
cached_dim_org = load_checkpoint(config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org cache")
cached_grid = load_checkpoint(config.paths.DATA_DIR + "/grid_training.parquet", "GRID cache")

if cached_dim_org is not None and cached_grid is not None:
    print("Using cached training data")
    dim_org_df = create_blocking_keys(cached_dim_org)
    grid_df = create_blocking_keys(cached_grid)
else:
    print("Loading fresh training data...")
    dim_org_df, grid_df = load_and_prepare_training_data(
        db,
        dim_org_sample=config.sampling.TRAINING_DIM_ORG_SAMPLE,
        grid_sample=config.sampling.TRAINING_GRID_SAMPLE
    )
    save_checkpoint(dim_org_df, config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org")
    save_checkpoint(grid_df, config.paths.DATA_DIR + "/grid_training.parquet", "GRID")

describe_dataframe(dim_org_df, "dim_organization")
describe_dataframe(grid_df, "GRID")


[16:06:24]  Loading checkpoint: dim_org cache
[16:06:24]    Loaded 109,851 rows
[16:06:24]  Loading checkpoint: GRID cache
[16:06:24]    Loaded 109,856 rows
Using cached training data
[16:06:24]  Creating blocking keys...
[16:06:25]    Added blocking keys to 109,851 records
[16:06:25]  Creating blocking keys...
[16:06:25]    Added blocking keys to 109,856 records

dim_organization
Shape: 109,851 rows x 20 columns

Column types:
  unique_id                      object            0.0% null
  name                           object            0.0% null
  name_clean                     object            0.0% null
  name_normalized                object            0.0% null
  name_prefix_5                  object            0.0% null
  name_prefix_10                 object            0.0% null
  all_names                      object            0.0% null
  org_type                       object            1.4% null
  country_code                   object            0.9% null
  city             

In [4]:
# Only keep records with BOTH identifiers
dim_org_df = dim_org_df[dim_org_df['ror_id'].notna() & dim_org_df['grid_id'].notna()].copy()
grid_df = grid_df[grid_df['ror_id'].notna() & grid_df['grid_id'].notna()].copy()

In [5]:
print(f"dim_org after filter: {len(dim_org_df):,}")
print(f"grid after filter: {len(grid_df):,}")
print(f"dim_org missing ROR: {dim_org_df['ror_id'].isna().sum()}")
print(f"dim_org missing GRID: {dim_org_df['grid_id'].isna().sum()}")
print(f"grid missing ROR: {grid_df['ror_id'].isna().sum()}")
print(f"grid missing GRID: {grid_df['grid_id'].isna().sum()}")

dim_org after filter: 100,253
grid after filter: 102,173
dim_org missing ROR: 0
dim_org missing GRID: 0
grid missing ROR: 0
grid missing GRID: 0


In [6]:
from data_prep import augment_with_aliases

dim_org_df, dim_swaps = augment_with_aliases(dim_org_df, swap_fraction=0.20, seed=42)
grid_df, grid_swaps = augment_with_aliases(grid_df, swap_fraction=0.20, seed=123)

print("\nSample dim_org swaps:")
for s in dim_swaps[:10]:
    print(f"  {s['original'][:45]:45} -> {s['swapped_to'][:45]}")

print("\nSample grid swaps:")
for s in grid_swaps[:10]:
    print(f"  {s['original'][:45]:45} -> {s['swapped_to'][:45]}")

[16:06:26]  Swapped names for 8,647 records (20%)
[16:06:27]  Swapped names for 4,136 records (20%)

Sample dim_org swaps:
  Faculdade de Medicina de Marília              -> Faculty of Medicine of Marília
  Ekaterinburg State Theatre Institute          -> Екатеринбургский государственный театральный 
  Türkiye Büyük Millet Meclisi                  -> Turkiye Buyuk Millet Meclisi
  MedTech CoRE                                  -> Consortium for Medical Device Technologies
  Gokhale Institute of Politics and Economics   -> Gokhale Institute
  Nordic Institute of Asian Studies             -> Nordisk Institut for Asien Studier
  Institut für Sonnenphysik                     -> Institute for Solar Physics
  Jiangsu Provincial Posts & Telecommunications -> JSPTPD
  Shahid Sadoughi University of Medical Science -> Shahid Sadoughi University of Medical Science
  Royal Alberta Museum                          -> Provincial Museum of Alberta

Sample grid swaps:
  Medtronic (India)                

In [7]:
# Add IDF-based distinctive tokens
print("ADDING IDF-BASED FEATURES")
print("=" * 50)

[dim_org_df, grid_df], idf_scores, corpus_stopwords = add_idf_based_features(
    [dim_org_df, grid_df],
    stopword_percentile=0.25
)

print(f"\nCorpus statistics:")
print(f"  Total unique tokens:   {len(idf_scores):,}")
print(f"  Auto-identified stopwords: {len(corpus_stopwords)}")

# Add token set features for containment detection
print(f"\nADDING TOKEN SET FEATURES")
dim_org_df = add_token_set_features(dim_org_df, stopwords=corpus_stopwords)
grid_df = add_token_set_features(grid_df, stopwords=corpus_stopwords)


ADDING IDF-BASED FEATURES
[16:06:27]  Adding IDF-based features to 2 DataFrames...
[16:06:27]  Computing token IDF statistics...
[16:06:27]    Computed IDF for 70,122 unique tokens
[16:06:27]    IDF range: 1.94 (most common) to 12.22 (most rare)
[16:06:27]    Auto-identified 18878 corpus stopwords (IDF <= 11.12)
[16:06:27]    Top 20 corpus stopwords: of, university, institute, and, hospital, for, research, de, center, foundation, college, national, medical, health, the, technology, association, centre, society, science
[16:06:27]  Adding distinctive tokens...
[16:06:27]    Loaded 34,655 geographic stopwords
[16:06:27]    Geographic stopwords: 34,655 (locations filtered out)
[16:06:27]    Distinctive token coverage: 99.9%
[16:06:27]  Adding distinctive tokens...
[16:06:27]    Geographic stopwords: 34,655 (locations filtered out)
[16:06:28]    Distinctive token coverage: 99.9%

Corpus statistics:
  Total unique tokens:   70,122
  Auto-identified stopwords: 18878

ADDING TOKEN SET FEATURE

In [8]:
# Create ground truth
positive_pairs, negative_pairs = create_ground_truth_pairs(dim_org_df, grid_df)

print("\nGROUND TRUTH SUMMARY")
print("=" * 50)
print(f"Positive pairs (same ROR): {len(positive_pairs):,}")
print(f"Negative pairs (diff ROR): {len(negative_pairs):,}")


[16:06:28]  Creating ground truth pairs from ROR and GRID matches...
[16:06:28]    dim_org with ROR: 100,253
[16:06:28]    GRID with ROR: 102,173
[16:06:28]    ROR positive pairs: 97,773
[16:06:28]    dim_org with GRID: 100,253
[16:06:28]    GRID with GRID: 102,173
[16:06:28]    GRID positive pairs: 97,773
[16:06:28]    Total positive pairs (deduplicated): 97,776
[16:06:28]    Negative pairs: 99,998

GROUND TRUTH SUMMARY
Positive pairs (same ROR): 97,776
Negative pairs (diff ROR): 99,998


In [9]:
# Define 9 Splink comparisons
print("DEFINING COMPARISONS")
print("=" * 50)

# 1. Name comparison with TF adjustments
name_comparison = cl.JaroWinklerAtThresholds(
    "name_normalized",
    [0.95, 0.88, 0.80],
).configure(term_frequency_adjustments=True)
print("  [x] Name comparison (Jaro-Winkler + TF adjustments)")

# 2. Phonetic comparison on distinctive token
phonetic_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("distinctive_soundex"),
        cll.ExactMatchLevel("distinctive_soundex"),
        cll.ElseLevel(),
    ],
    output_column_name="phonetic_match",
)
print("  [x] Phonetic comparison (on distinctive token Soundex)")

# 3. Distinctive tokens comparison
distinctive_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("distinctive_tokens"),
        {
            "sql_condition": "array_length(distinctive_tokens_l, 1) = 0 OR array_length(distinctive_tokens_r, 1) = 0",
            "label_for_charts": "Empty array (short name)",
        },
        cll.ArrayIntersectLevel("distinctive_tokens", min_intersection=2),
        cll.ArrayIntersectLevel("distinctive_tokens", min_intersection=1),
        cll.ElseLevel(),
    ],
    output_column_name="distinctive_match",
)
print("  [x] Distinctive tokens comparison")

# 4. Token overlap comparison
token_overlap_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("distinctive_tokens"),
        cll.ArrayIntersectLevel("distinctive_tokens", min_intersection=2),
        cll.ArrayIntersectLevel("distinctive_tokens", min_intersection=1),
        cll.ElseLevel(),
    ],
    output_column_name="token_overlap",
)
print("  [x] Token overlap comparison")

# 5. Containment detection
containment_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("name_tokens"),
        {
            "sql_condition": "list_sort(name_tokens_l) = list_sort(name_tokens_r)",
            "label_for_charts": "Exact token match",
        },
        {
            "sql_condition": """
                ABS(len(name_tokens_l) - len(name_tokens_r)) <= 1
                AND len(list_intersect(name_tokens_l, name_tokens_r)) >= 
                    GREATEST(len(name_tokens_l), len(name_tokens_r)) - 1
            """,
            "label_for_charts": "Similar tokens (diff <= 1)",
        },
        {
            "sql_condition": """
                (len(name_tokens_l) <= 2 OR len(name_tokens_r) <= 2)
                AND len(list_intersect(name_tokens_l, name_tokens_r)) >= 1
            """,
            "label_for_charts": "Short name with overlap",
        },
        {
            "sql_condition": """
                (list_sort(list_intersect(name_tokens_l, name_tokens_r)) = list_sort(name_tokens_l)
                 AND len(name_tokens_l) < len(name_tokens_r))
                OR
                (list_sort(list_intersect(name_tokens_l, name_tokens_r)) = list_sort(name_tokens_r)
                 AND len(name_tokens_r) < len(name_tokens_l))
            """,
            "label_for_charts": "Subset containment (suspicious)",
        },
        cll.ElseLevel(),
    ],
    output_column_name="containment_check",
)
print("  [x] Containment detection")

# 6. First token comparison with Jaro-Winkler
first_token_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("distinctive_token"),
        cll.ExactMatchLevel("distinctive_token"),
        cll.JaroWinklerLevel("distinctive_token", 0.9),
        cll.JaroWinklerLevel("distinctive_token", 0.8),
        cll.ElseLevel(),
    ],
    output_column_name="first_token_match",
)
print("  [x] First token comparison")

# 7. Alias matching
alias_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("all_names"),
        {
            "sql_condition": """
                list_contains(list_transform(all_names_r, x -> LOWER(x)), LOWER(name_l)) OR 
                list_contains(list_transform(all_names_l, x -> LOWER(x)), LOWER(name_r))
            """,
            "label_for_charts": "Name matches alias (exact)",
        },
        cll.ElseLevel(),
    ],
    output_column_name="alias_match",
)
print("  [x] Alias matching")

# 8. Country comparison
country_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("country_code"),
        cll.ExactMatchLevel("country_code"),
        cll.ElseLevel(),
    ],
    output_column_name="country_match",
)
print("  [x] Country comparison")

# 9. City comparison
city_comparison = cl.CustomComparison(
    comparison_levels=[
        cll.NullLevel("city"),
        cll.ExactMatchLevel("city"),
        cll.JaroWinklerLevel("city", 0.9),
        cll.ElseLevel(),
    ],
    output_column_name="city_match",
)
print("  [x] City comparison")

comparisons = [
    name_comparison,
    phonetic_comparison,
    distinctive_comparison,
    token_overlap_comparison,
    containment_comparison,
    first_token_comparison,
    alias_comparison,
    country_comparison,
    city_comparison,
]


DEFINING COMPARISONS
  [x] Name comparison (Jaro-Winkler + TF adjustments)
  [x] Phonetic comparison (on distinctive token Soundex)
  [x] Distinctive tokens comparison
  [x] Token overlap comparison
  [x] Containment detection
  [x] First token comparison
  [x] Alias matching
  [x] Country comparison
  [x] City comparison


In [10]:
# Define 4 blocking rules
print("DEFINING BLOCKING RULES")
print("=" * 50)
blocking_rules = [
    # Tier 1: High precision (when geo exists)
    "l.name_normalized = r.name_normalized AND l.country_code = r.country_code",
    "substr(l.name_normalized, 1, 25) = substr(r.name_normalized, 1, 25) AND l.country_code = r.country_code",
    "l.distinctive_token = r.distinctive_token AND l.city = r.city",
    
    # Tier 2: Fallback for missing geo (very strict name matching)
    "l.name_normalized = r.name_normalized AND length(l.name_normalized) >= 30",  # Long exact names only
]
for i, rule in enumerate(blocking_rules):
    print(f"  [{i+1}] {rule}")


DEFINING BLOCKING RULES
  [1] l.name_normalized = r.name_normalized AND l.country_code = r.country_code
  [2] substr(l.name_normalized, 1, 25) = substr(r.name_normalized, 1, 25) AND l.country_code = r.country_code
  [3] l.distinctive_token = r.distinctive_token AND l.city = r.city
  [4] l.name_normalized = r.name_normalized AND length(l.name_normalized) >= 30


In [11]:
# Create settings and linker
settings = SettingsCreator(
    link_type="link_only",
    unique_id_column_name="unique_id",
    comparisons=comparisons,
    blocking_rules_to_generate_predictions=blocking_rules,
)

print(f"\nSettings: {len(comparisons)} comparisons, {len(blocking_rules)} blocking rules")

with Timer("Initializing Linker"):
    linker = Linker([dim_org_df, grid_df], settings, db_api=DuckDBAPI())
print("Linker initialized")



Settings: 9 comparisons, 4 blocking rules
[16:06:29]  Starting: Initializing Linker
[16:06:29]  Completed: Initializing Linker (0.2s)
Linker initialized


In [12]:
# Estimate u-probabilities
with Timer("Estimating u-probabilities"):
    linker.training.estimate_u_using_random_sampling(max_pairs=config.sampling.U_PROBABILITY_MAX_PAIRS)


----- Estimating u probabilities using random sampling -----


[16:06:30]  Starting: Estimating u-probabilities



Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - name_normalized (no m values are trained).
    - phonetic_match (no m values are trained).
    - distinctive_match (no m values are trained).
    - token_overlap (no m values are trained).
    - containment_check (no m values are trained).
    - first_token_match (no m values are trained).
    - alias_match (no m values are trained).
    - country_match (no m values are trained).
    - city_match (no m values are trained).


[16:06:40]  Completed: Estimating u-probabilities (10.0s)


In [13]:
# EM Training Stage 1
with Timer("EM Training - Stage 1 (name_normalized)"):
    linker.training.estimate_parameters_using_expectation_maximisation(
        block_on("name_normalized"), fix_u_probabilities=True
    )


[16:06:40]  Starting: EM Training - Stage 1 (name_normalized)



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."name_normalized" = r."name_normalized"

Parameter estimates will be made for the following comparison(s):
    - phonetic_match
    - distinctive_match
    - token_overlap
    - containment_check
    - first_token_match
    - alias_match
    - country_match
    - city_match

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - name_normalized

Level All other comparisons on comparison phonetic_match not observed in dataset, unable to train m value

Level All other comparisons on comparison distinctive_match not observed in dataset, unable to train m value

Level Similar tokens (diff <= 1) on comparison containment_check not observed in dataset, unable to train m value

Level Short name with overlap on comparison containment_check not observed in dataset, unable to train m value

Level Subset containment (suspicious) on 

[16:06:42]  Completed: EM Training - Stage 1 (name_normalized) (2.0s)


In [14]:
# EM Training Stage 2
with Timer("EM Training - Stage 2 (country + name_prefix_5)"):
    linker.training.estimate_parameters_using_expectation_maximisation(
        "l.country_code = r.country_code AND l.name_prefix_5 = r.name_prefix_5",
        fix_u_probabilities=True
    )



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l.country_code = r.country_code AND l.name_prefix_5 = r.name_prefix_5

Parameter estimates will be made for the following comparison(s):
    - name_normalized
    - phonetic_match
    - distinctive_match
    - token_overlap
    - containment_check
    - first_token_match
    - alias_match
    - city_match

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - country_match


[16:06:42]  Starting: EM Training - Stage 2 (country + name_prefix_5)



Iteration 1: Largest change in params was -0.589 in the m_probability of containment_check, level `Exact token match`
Iteration 2: Largest change in params was -0.556 in the m_probability of first_token_match, level `Exact match on distinctive_token`
Iteration 3: Largest change in params was 0.357 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 4: Largest change in params was 0.322 in probability_two_random_records_match
Iteration 5: Largest change in params was 0.0869 in probability_two_random_records_match
Iteration 6: Largest change in params was 0.0305 in the m_probability of name_normalized, level `All other comparisons`
Iteration 7: Largest change in params was 0.0138 in the m_probability of name_normalized, level `All other comparisons`
Iteration 8: Largest change in params was 0.00685 in the m_probability of name_normalized, level `All other comparisons`
Iteration 9: Largest change in params was 0.00359 in the m_probability of name_normalized

[16:06:57]  Completed: EM Training - Stage 2 (country + name_prefix_5) (14.9s)


In [15]:
# EM Training Stage 3
with Timer("EM Training - Stage 3 (distinctive_token)"):
    linker.training.estimate_parameters_using_expectation_maximisation(
        "l.distinctive_token = r.distinctive_token", fix_u_probabilities=True
    )



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l.distinctive_token = r.distinctive_token

Parameter estimates will be made for the following comparison(s):
    - name_normalized
    - phonetic_match
    - distinctive_match
    - token_overlap
    - containment_check
    - alias_match
    - country_match
    - city_match

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - first_token_match


[16:06:57]  Starting: EM Training - Stage 3 (distinctive_token)



Level All other comparisons on comparison phonetic_match not observed in dataset, unable to train m value

Level Empty array (short name) on comparison distinctive_match not observed in dataset, unable to train m value

Level All other comparisons on comparison distinctive_match not observed in dataset, unable to train m value

Level All other comparisons on comparison token_overlap not observed in dataset, unable to train m value

Iteration 1: Largest change in params was -0.909 in the m_probability of phonetic_match, level `All other comparisons`
Iteration 2: Largest change in params was 4.51e-06 in probability_two_random_records_match

EM converged after 2 iterations
m probability not trained for phonetic_match - All other comparisons (comparison vector value: 0). This usually means the comparison level was never observed in the training data.
m probability not trained for distinctive_match - Empty array (short name) (comparison vector value: 3). This usually means the comparison l

[16:07:01]  Completed: EM Training - Stage 3 (distinctive_token) (4.3s)


In [16]:
# EM Training Stage 4
with Timer("EM Training - Stage 4 (city + name_prefix_5)"):
    linker.training.estimate_parameters_using_expectation_maximisation(
        "l.city = r.city AND l.name_prefix_5 = r.name_prefix_5", fix_u_probabilities=True
    )



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l.city = r.city AND l.name_prefix_5 = r.name_prefix_5

Parameter estimates will be made for the following comparison(s):
    - name_normalized
    - phonetic_match
    - distinctive_match
    - token_overlap
    - containment_check
    - first_token_match
    - alias_match
    - country_match

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - city_match


[16:07:01]  Starting: EM Training - Stage 4 (city + name_prefix_5)



Iteration 1: Largest change in params was -0.684 in the m_probability of phonetic_match, level `Exact match on distinctive_soundex`
Iteration 2: Largest change in params was 0.135 in probability_two_random_records_match
Iteration 3: Largest change in params was 0.0646 in the m_probability of name_normalized, level `All other comparisons`
Iteration 4: Largest change in params was 0.0621 in the m_probability of name_normalized, level `All other comparisons`
Iteration 5: Largest change in params was 0.0657 in probability_two_random_records_match
Iteration 6: Largest change in params was 0.0611 in probability_two_random_records_match
Iteration 7: Largest change in params was 0.0431 in probability_two_random_records_match
Iteration 8: Largest change in params was 0.0231 in probability_two_random_records_match
Iteration 9: Largest change in params was 0.0103 in probability_two_random_records_match
Iteration 10: Largest change in params was 0.00431 in probability_two_random_records_match
Ite

[16:07:06]  Completed: EM Training - Stage 4 (city + name_prefix_5) (4.7s)


In [17]:
# EM Training Stage 5
with Timer("EM Training - Stage 5 (country + city)"):
    linker.training.estimate_parameters_using_expectation_maximisation(
        "l.country_code = r.country_code AND l.city = r.city", fix_u_probabilities=True
    )



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l.country_code = r.country_code AND l.city = r.city

Parameter estimates will be made for the following comparison(s):
    - name_normalized
    - phonetic_match
    - distinctive_match
    - token_overlap
    - containment_check
    - first_token_match
    - alias_match

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - country_match
    - city_match


[16:07:06]  Starting: EM Training - Stage 5 (country + city)



Iteration 1: Largest change in params was -0.386 in the m_probability of phonetic_match, level `Exact match on distinctive_soundex`
Iteration 2: Largest change in params was -0.14 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 3: Largest change in params was -0.141 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 4: Largest change in params was -0.126 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 5: Largest change in params was -0.0874 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 6: Largest change in params was -0.0496 in the m_probability of distinctive_match, level `All other comparisons`
Iteration 7: Largest change in params was -0.0258 in the m_probability of token_overlap, level `All other comparisons`
Iteration 8: Largest change in params was -0.0148 in the m_probability of token_overlap, level `All other comparisons`
Iteration 9: Large

[16:08:13]  Completed: EM Training - Stage 5 (country + city) (67.2s)


In [18]:
# Set conservative prior
print("PRIOR PROBABILITY")
n_positive = len(positive_pairs)
n_total_possible = len(dim_org_df) * len(grid_df)
print(f"Known positive pairs:     {n_positive:,}")
print(f"Total possible pairs:     {n_total_possible:,}")

conservative_prior = 1e-7
linker._settings_obj._probability_two_random_records_match = conservative_prior
print(f"Using conservative prior: {conservative_prior:.8f}")


PRIOR PROBABILITY
Known positive pairs:     97,776
Total possible pairs:     10,243,149,769
Using conservative prior: 0.00000010


In [19]:
linker.visualisations.match_weights_chart()


alt.VConcatChart(...)

In [20]:
# Generate predictions and validate
with Timer("Generating predictions"):
    predictions = linker.inference.predict(threshold_match_probability=config.matching.THRESHOLD_PREDICTION)
    predictions_df = predictions.as_pandas_dataframe()

print(f"\nTotal predictions: {len(predictions_df):,}")

bins = [0, 0.5, 0.7, 0.85, 0.95, 1.0]
labels = ['<0.5', '0.5-0.7', '0.7-0.85', '0.85-0.95', '0.95-1.0']
predictions_df['score_bin'] = pd.cut(predictions_df['match_probability'], bins=bins, labels=labels)
print("\nMatch probability distribution:")
print(predictions_df['score_bin'].value_counts().sort_index())


Blocking time: 0.19 seconds


[16:08:13]  Starting: Generating predictions


Predict time: 1.07 seconds


[16:08:16]  Completed: Generating predictions (2.5s)

Total predictions: 151,989

Match probability distribution:
score_bin
<0.5              0
0.5-0.7          28
0.7-0.85      11139
0.85-0.95       820
0.95-1.0     140002
Name: count, dtype: int64


In [21]:
# Add ground truth labels and calculate metrics

# Merge identifier columns into predictions for direct verification
predictions_df = predictions_df.merge(
    dim_org_df[['unique_id', 'ror_id', 'grid_id']].rename(columns={'ror_id': 'ror_l', 'grid_id': 'grid_l'}),
    left_on='unique_id_l', right_on='unique_id', how='left'
).drop(columns=['unique_id'])

predictions_df = predictions_df.merge(
    grid_df[['unique_id', 'ror_id', 'grid_id']].rename(columns={'ror_id': 'ror_r', 'grid_id': 'grid_r'}),
    left_on='unique_id_r', right_on='unique_id', how='left'
).drop(columns=['unique_id'])

# Direct ground truth: match if same ROR OR same GRID
predictions_df['is_true_match'] = (
    (predictions_df['ror_l'] == predictions_df['ror_r']) | 
    (predictions_df['grid_l'] == predictions_df['grid_r'])
)

print("Ground Truth Distribution:")
print(predictions_df['is_true_match'].value_counts())

# Calculate metrics on ALL predictions
print("\nMETRICS:")
for thresh in [0.5, 0.85, 0.95]:
    pred_positive = predictions_df['match_probability'] >= thresh
    actual_positive = predictions_df['is_true_match']
    tp = (actual_positive & pred_positive).sum()
    fp = (~actual_positive & pred_positive).sum()
    fn = (actual_positive & ~pred_positive).sum()
    tn = (~actual_positive & ~pred_positive).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"Threshold >= {thresh}: P={precision:.4f} R={recall:.4f} F1={f1:.4f} (TP={tp:,} FP={fp:,} FN={fn:,})")

Ground Truth Distribution:
is_true_match
True     87477
False    64512
Name: count, dtype: int64

METRICS:
Threshold >= 0.5: P=0.5755 R=1.0000 F1=0.7306 (TP=87,477 FP=64,512 FN=0)
Threshold >= 0.85: P=0.6212 R=1.0000 F1=0.7663 (TP=87,477 FP=53,345 FN=0)
Threshold >= 0.95: P=0.6248 R=1.0000 F1=0.7691 (TP=87,477 FP=52,525 FN=0)


In [22]:
# Diagnose false positives
fp_df = predictions_df[(predictions_df['match_probability'] >= 0.5) & (~predictions_df['is_true_match'])].copy()

print(f"Total FP: {len(fp_df):,}")
print(f"\nFP score distribution:")
print(fp_df['match_probability'].describe())

# Look at high-confidence false positives (most concerning)
high_conf_fp = fp_df[fp_df['match_probability'] >= 0.95].sort_values('match_probability', ascending=False)
print(f"\nHigh confidence FP (>=0.95): {len(high_conf_fp):,}")

# Sample some to inspect
cols = ['name_l', 'name_r', 'country_code_l', 'country_code_r', 'city_l', 'city_r', 
        'match_probability', 'ror_l', 'ror_r', 'grid_l', 'grid_r']
print("\nSample high-confidence FPs:")
display(high_conf_fp[cols].head(20))

# Check if they share similar names but different orgs
print("\n--- Name similarity in FPs ---")
for _, row in high_conf_fp.head(10).iterrows():
    print(f"\n{row['name_l'][:50]:50} vs {row['name_r'][:50]}")
    print(f"  Country: {row['country_code_l']} vs {row['country_code_r']}")
    print(f"  City: {row['city_l']} vs {row['city_r']}")
    print(f"  Score: {row['match_probability']:.3f}")

Total FP: 64,512

FP score distribution:
count    64512.000000
mean         0.967842
std          0.069186
min          0.533267
25%          0.998736
50%          0.999999
75%          1.000000
max          1.000000
Name: match_probability, dtype: float64

High confidence FP (>=0.95): 52,525

Sample high-confidence FPs:


,name_l,name_r,country_code_l,country_code_r,city_l,city_r,match_probability,ror_l,ror_r,grid_l,grid_r
35146,Sanford Medical Center,Sanford Medical Center,US,US,Bismarck,Fargo,1.0,https://ror.org/037xpb040,https://ror.org/03byzcq48,grid.437741.2,grid.429398.d
108323,AbbVie (Netherlands),AbbVie (Netherlands),NL,NL,Zwolle,Zwolle,1.0,https://ror.org/020j24g35,https://ror.org/01qa0ew63,grid.488252.1,grid.482390.4
8896,Institut de Recherche pour le Développement,Institut de Recherche pour le Développement,SN,TN,Dakar,Tunis,1.0,https://ror.org/015q23935,https://ror.org/044vzpb64,grid.418291.7,grid.463365.1
44460,Children's Oncology Group,Children's Oncology Group,CH,US,Monrovia,Monrovia,1.0,https://ror.org/037wy0p68,https://ror.org/03yyg2352,grid.476225.0,grid.428204.8
12621,European Molecular Biology Laboratory,European Molecular Biology Laboratory,ES,DE,Barcelona,Heidelberg,1.0,https://ror.org/010jaxs89,https://ror.org/03mstc592,grid.495034.f,grid.4709.a
113044,Institute of Marine Geology and Geophysics,Institute of Marine Geology and Geophysics,VN,RU,Hanoi,Yuzhno-Sakhalinsk,1.0,https://ror.org/057as5086,https://ror.org/021923v79,grid.472705.3,grid.494538.7
44500,Instituto Federal de Educação,Instituto Federal de Educação,BR,BR,Manaus,Blumenau,1.0,https://ror.org/045nsn047,https://ror.org/02f8h1m78,grid.472923.9,grid.454337.2
70450,Institute of Molecular Biology,Institute of Molecular Biology,TW,BG,Taipei,Sofia,1.0,https://ror.org/047sbcx71,https://ror.org/00t7c6f62,grid.506935.c,grid.425038.8
102435,Institut de Recherche pour le Développement,Institut de Recherche pour le Développement,BJ,TN,Cotonou,Tunis,1.0,https://ror.org/032qezt74,https://ror.org/044vzpb64,grid.473220.0,grid.463365.1
70448,National Center for Supercomputing Applications,National Center for Supercomputing Applications,US,BG,Urbana,Sofia,1.0,https://ror.org/03r10zj06,https://ror.org/03g9ch715,grid.505692.d,grid.432563.5



--- Name similarity in FPs ---

Sanford Medical Center                             vs Sanford Medical Center
  Country: US vs US
  City: Bismarck vs Fargo
  Score: 1.000

AbbVie (Netherlands)                               vs AbbVie (Netherlands)
  Country: NL vs NL
  City: Zwolle vs Zwolle
  Score: 1.000

Institut de Recherche pour le Développement        vs Institut de Recherche pour le Développement
  Country: SN vs TN
  City: Dakar vs Tunis
  Score: 1.000

Children's Oncology Group                          vs Children's Oncology Group
  Country: CH vs US
  City: Monrovia vs Monrovia
  Score: 1.000

European Molecular Biology Laboratory              vs European Molecular Biology Laboratory
  Country: ES vs DE
  City: Barcelona vs Heidelberg
  Score: 1.000

Institute of Marine Geology and Geophysics         vs Institute of Marine Geology and Geophysics
  Country: VN vs RU
  City: Hanoi vs Yuzhno-Sakhalinsk
  Score: 1.000

Instituto Federal de Educação                      vs Institut

In [26]:
# Pick a sample FP to investigate (e.g., the AbbVie case)
sample_fp = high_conf_fp.iloc[1]  # or filter to a specific row

# Get the unique IDs
uid_l = sample_fp['unique_id_l']
uid_r = sample_fp['unique_id_r']

print(f"=== LEFT RECORD (dim_org) ===")
print(f"unique_id: {uid_l}")
left_record = dim_org_df[dim_org_df['unique_id'] == uid_l].T
display(left_record)

print(f"\n=== RIGHT RECORD (grid) ===")
print(f"unique_id: {uid_r}")
right_record = grid_df[grid_df['unique_id'] == uid_r].T
display(right_record)

print(f"\n=== COMPARISON ===")
print(f"Match probability: {sample_fp['match_probability']:.4f}")
print(f"ROR match: {sample_fp['ror_l']} == {sample_fp['ror_r']} ? {sample_fp['ror_l'] == sample_fp['ror_r']}")
print(f"GRID match: {sample_fp['grid_l']} == {sample_fp['grid_r']} ? {sample_fp['grid_l'] == sample_fp['grid_r']}")

=== LEFT RECORD (dim_org) ===
unique_id: dim_ASC-OR-0000000098438-1.0-1724880263


,73623
unique_id,dim_ASC-OR-0000000098438-1.0-1724880263
name,AbbVie (Netherlands)
name_clean,abbvie (netherlands)
name_normalized,abbvie
name_prefix_5,abbvi
name_prefix_10,abbvie
all_names,"[Actavis, Allergan (Netherlands), AbbVie (Netherlands)]"
org_type,company
country_code,NL
city,Zwolle



=== RIGHT RECORD (grid) ===
unique_id: grid_grid.482390.4


,69371
unique_id,grid_grid.482390.4
name,AbbVie (Netherlands)
name_clean,abbvie (netherlands)
name_normalized,abbvie
name_prefix_5,abbvi
name_prefix_10,abbvie
all_names,[AbbVie (Netherlands)]
org_type,[Company]
country_code,NL
city,Zwolle



=== COMPARISON ===
Match probability: 1.0000
ROR match: https://ror.org/020j24g35 == https://ror.org/01qa0ew63 ? False
GRID match: grid.488252.1 == grid.482390.4 ? False


In [23]:
# 1. Check what blocking rules generated
print("Predictions by blocking rule:")
if 'match_key' in predictions_df.columns:
    print(predictions_df['match_key'].value_counts())

# 2. Sample high-confidence FPs to see WHY they match
fp_high = predictions_df[(predictions_df['match_probability'] >= 0.95) & (~predictions_df['is_true_match'])]
print(f"\nHigh-conf FPs: {len(fp_high):,}")

sample = fp_high.sample(10, random_state=42)
for _, row in sample.iterrows():
    print(f"\n{row['name_l'][:50]} vs {row['name_r'][:50]}")
    print(f"  Country: {row.get('country_code_l')} vs {row.get('country_code_r')}")
    print(f"  City: {row.get('city_l')} vs {row.get('city_r')}")
    print(f"  Score: {row['match_probability']:.3f}")

Predictions by blocking rule:
match_key
0    85263
2    39575
1    26224
3      927
Name: count, dtype: int64

High-conf FPs: 52,525

Directorate-General for Education, Youth, Sport an vs Directorate-General for Migration and Home Affairs
  Country: BE vs BE
  City: Brussels vs Brussels
  Score: 1.000

Ministry of Higher Education and Scientific Resear vs Ministry of Higher Education and Scientific Resear
  Country: LY vs DZ
  City: Tripoli vs Algiers
  Score: 1.000

Department of Education Shandong Province vs Shandong Academy of Medical Science
  Country: CN vs CN
  City: Jinan vs Jinan
  Score: 1.000

Second Affiliated Hospital of Luohe Medical Colleg vs Second Affiliated Hospital of Chengdu University o
  Country: CN vs CN
  City: Luohe vs Chengdu
  Score: 1.000

Centro Nacional de Alta Tecnología de Costa Rica vs Business University of Costa Rica
  Country: CR vs CR
  City: San José vs San José
  Score: 1.000

Hunan Provincial Center for Disease Control and Pr vs Hunan University 

In [24]:
# What are your current blocking rules?
print("Blocking rules:")
for i, rule in enumerate(settings.blocking_rules_to_generate_predictions):
    print(f"  [{i}] {rule}")

Blocking rules:
  [0] l.name_normalized = r.name_normalized AND l.country_code = r.country_code
  [1] substr(l.name_normalized, 1, 25) = substr(r.name_normalized, 1, 25) AND l.country_code = r.country_code
  [2] l.distinctive_token = r.distinctive_token AND l.city = r.city
  [3] l.name_normalized = r.name_normalized AND length(l.name_normalized) >= 30


In [25]:
same_country_fp = high_conf_fp[high_conf_fp['country_code_l'] == high_conf_fp['country_code_r']]
print(f"High-conf FPs with same country: {len(same_country_fp):,} / {len(high_conf_fp):,}")

High-conf FPs with same country: 50,753 / 52,525


In [ ]:
# After predictions, cluster into groups
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    predictions,
    threshold_match_probability=0.85
)
clusters_df = clusters.as_pandas_dataframe()

In [ ]:
clusters_df

In [ ]:
# Check distribution of cluster sizes
cluster_sizes = clusters_df.groupby('cluster_id').size().reset_index(name='count')
print(f"Total clusters: {len(cluster_sizes):,}")
print(f"\nCluster size distribution:")
print(cluster_sizes['count'].describe())

# Show large clusters (potential chains/systems)
large_clusters = cluster_sizes[cluster_sizes['count'] > 2].sort_values('count', ascending=False)
print(f"\nClusters with 3+ members: {len(large_clusters):,}")

In [ ]:
# Top 10 largest clusters
top_clusters = cluster_sizes.nlargest(10, 'count')
print("Largest clusters:")
print(top_clusters)

# Inspect the biggest one
biggest_cluster_id = top_clusters.iloc[0]['cluster_id']
biggest = clusters_df[clusters_df['cluster_id'] == biggest_cluster_id]
print(f"\nBiggest cluster ({len(biggest)} members):")
print(biggest[['name', 'country_code', 'city']].head(20))

In [ ]:
# Save model
import json
from pathlib import Path

model_path = config.paths.MODEL_FILE
Path(model_path).parent.mkdir(parents=True, exist_ok=True)

model_json = linker.misc.save_model_to_json()
model_json['training_metadata'] = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'dim_org_records': len(dim_org_df),
    'grid_records': len(grid_df),
    'positive_pairs': len(positive_pairs),
    'negative_pairs': len(negative_pairs),
}

with open(model_path, 'w') as f:
    json.dump(model_json, f, indent=2)

print(f"Model saved to: {model_path}")
print("\n" + "=" * 50)
print("TRAINING COMPLETE")
print("=" * 50)
print(f"dim_org: {len(dim_org_df):,} | GRID: {len(grid_df):,}")
print(f"Predictions: {len(predictions_df):,}")
print(f"High confidence (>0.95): {(predictions_df['match_probability'] > 0.95).sum():,}")

db.close()
